<a href="https://colab.research.google.com/github/philipgp/MusicPlus/blob/master/MusicRecommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Path to an example CSV file in your environment
csv_file_path = '/content/musick_feature_store_music_play_progress_202605170210.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(csv_file_path)

# Display the first few rows of the DataFrame
print(df)

In [65]:
df["hours_since_last_play"] = df["hours_since_last_play"].fillna(999)
df["last_play_pct"] = df["last_play_pct"].fillna(0)
df["session_recent_volatility"] = df["session_recent_volatility"].fillna(0)
df["bluetoothDevice"] = df["bluetoothDevice"].fillna("UNKNOWN")
df["bluetoothDevice"] = df["bluetoothDevice"].replace("", "SPEAKER")

  # bpm
df["bpm"] = df["bpm"].fillna(df["bpm"].mean())

X=df.iloc[:,4:]
y=(df.iloc[:,3] >= 30).astype(int)
print(df.isnull().sum()[df.isnull().sum() > 0])


Series([], dtype: int64)


In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE



In [66]:
X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.2, random_state=42, stratify=y  # stratify preserves 80/20 ratio in splits
  )


In [67]:
preprocessor = ColumnTransformer([
      ("ohe", OneHotEncoder(handle_unknown="ignore"), ["bluetoothDevice"]),
      ("passthrough", "passthrough", [c for c in X_train.columns if c != "bluetoothDevice"]),
  ])
model = Pipeline([

   ("smote", SMOTE(random_state=42)),
      ("classifier", RandomForestClassifier(
          n_estimators=200,
         class_weight="balanced",   # back to balanced

          random_state=42,
          n_jobs=-1,
      )),
  ])


In [ ]:

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.83      0.93      0.88       725
           1       0.62      0.39      0.48       226

    accuracy                           0.80       951
   macro avg       0.73      0.66      0.68       951
weighted avg       0.78      0.80      0.78       951



In [ ]:

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


In [69]:
y_proba = model.predict_proba(X_test)[:, 1]

for threshold in [0.3, 0.35, 0.4, 0.45, 0.5]:
      y_pred = (y_proba >= threshold).astype(int)
      report = classification_report(y_test, y_pred, output_dict=True)
      print(f"threshold={threshold:.2f}  precision={report['1']['precision']:.2f}  recall={report['1']['recall']:.2f}  f1={report['1']['f1-score']:.2f}")


threshold=0.30  precision=0.43  recall=0.69  f1=0.53
threshold=0.35  precision=0.48  recall=0.62  f1=0.54
threshold=0.40  precision=0.53  recall=0.55  f1=0.54
threshold=0.45  precision=0.58  recall=0.47  f1=0.52
threshold=0.50  precision=0.62  recall=0.39  f1=0.48
